In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers datasets torch scikit-learn streamlit -q

In [ ]:
!git clone https://github.com/google-research/google-research.git

In [ ]:
!git clone https://github.com/google-research/google-research.git

In [ ]:
!mkdir -p /content/goemotions_split

!wget -O /content/goemotions_split/train.tsv https://raw.githubusercontent.com/google-research/google-research/master/goemotions/data/train.tsv
!wget -O /content/goemotions_split/dev.tsv https://raw.githubusercontent.com/google-research/google-research/master/goemotions/data/dev.tsv
!wget -O /content/goemotions_split/test.tsv https://raw.githubusercontent.com/google-research/google-research/master/goemotions/data/test.tsv

In [ ]:
train_df = pd.read_csv(
    "/content/goemotions_split/train.tsv",
    sep="\t",
    header=None,
    names=["text", "labels", "id"]
)

dev_df = pd.read_csv(
    "/content/goemotions_split/dev.tsv",
    sep="\t",
    header=None,
    names=["text", "labels", "id"]
)

test_df = pd.read_csv(
    "/content/goemotions_split/test.tsv",
    sep="\t",
    header=None,
    names=["text", "labels", "id"]
)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

train_df.head()

In [ ]:
emotion_labels = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness", "optimism",
    "pride", "realization", "relief", "remorse", "sadness",
    "surprise", "neutral"
]

num_classes = len(emotion_labels)

print("Number of classes:", num_classes)

In [ ]:
def get_first_label(label_string):
    return int(str(label_string).split(",")[0])

train_df["label"] = train_df["labels"].apply(get_first_label)
dev_df["label"] = dev_df["labels"].apply(get_first_label)
test_df["label"] = test_df["labels"].apply(get_first_label)

train_df.head()

In [ ]:
!pip install transformers scikit-learn -q

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 20000
MAX_LEN = 80
BATCH_SIZE = 64

X_train = train_df["text"].astype(str).values
y_train = train_df["label"].values

X_test = test_df["text"].astype(str).values
y_test = test_df["label"].values

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

X_train_tensor = torch.tensor(X_train_pad, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_pad, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return self.texts[index], self.labels[index]

train_dataset = EmotionDataset(X_train_tensor, y_train_tensor)
test_dataset = EmotionDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
import torch.nn as nn

class BiLSTMEmotionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(BiLSTMEmotionClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        final_output = lstm_out[:, -1, :]
        output = self.dropout(final_output)
        output = self.fc(output)
        return output

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

bilstm_model = BiLSTMEmotionClassifier(
    vocab_size=MAX_WORDS,
    embedding_dim=128,
    hidden_dim=128,
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=0.001)

EPOCHS = 3

for epoch in range(EPOCHS):
    bilstm_model.train()
    total_loss = 0

    for texts, labels in train_loader:
        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = bilstm_model(texts)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {avg_loss:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

bilstm_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.to(device)
        labels = labels.to(device)

        outputs = bilstm_model(texts)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

bilstm_accuracy = accuracy_score(all_labels, all_preds)
bilstm_precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
bilstm_recall = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
bilstm_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

print("BiLSTM Accuracy:", bilstm_accuracy)
print("BiLSTM Precision:", bilstm_precision)
print("BiLSTM Recall:", bilstm_recall)
print("BiLSTM F1 Score:", bilstm_f1)

In [ ]:
from transformers import AutoTokenizer

bert_model_name = "distilbert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

train_encodings = bert_tokenizer(
    list(train_df["text"].astype(str)),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = bert_tokenizer(
    list(test_df["text"].astype(str)),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
class BertEmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: torch.tensor(value[index])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

bert_train_dataset = BertEmotionDataset(train_encodings, train_df["label"].values)
bert_test_dataset = BertEmotionDataset(test_encodings, test_df["label"].values)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_model_name,
    num_labels=num_classes
)

training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(.   n v b=-091``03211
    args=training_args,
    train_dataset=bert_train_dataset,
    eval_dataset=bert_test_dataset
)

trainer.train()

In [ ]:
bert_predictions = trainer.predict(bert_test_dataset)

bert_preds = np.argmax(bert_predictions.predictions, axis=1)
bert_labels = bert_predictions.label_ids

bert_accuracy = accuracy_score(bert_labels, bert_preds)
bert_precision = precision_score(bert_labels, bert_preds, average="weighted", zero_division=0)
bert_recall = recall_score(bert_labels, bert_preds, average="weighted", zero_division=0)
bert_f1 = f1_score(bert_labels, bert_preds, average="weighted", zero_division=0)

print("BERT Accuracy:", bert_accuracy)
print("BERT Precision:", bert_precision)
print("BERT Recall:", bert_recall)
print("BERT F1 Score:", bert_f1)

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["BiLSTM", "BERT"],
    "Accuracy": [bilstm_accuracy, bert_accuracy],
    "Precision": [bilstm_precision, bert_precision],
    "Recall": [bilstm_recall, bert_recall],
    "F1 Score": [bilstm_f1, bert_f1]
})

results

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import pickle

SAVE_DIR = "/content/drive/MyDrive/AI_Therapist_Models"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(
    bilstm_model.state_dict(),
    f"{SAVE_DIR}/bilstm_model.pth"
)

with open(f"{SAVE_DIR}/bilstm_tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

with open(f"{SAVE_DIR}/emotion_labels.pkl", "wb") as file:
    pickle.dump(emotion_labels, file)

bert_model.save_pretrained(f"{SAVE_DIR}/bert_model")
bert_tokenizer.save_pretrained(f"{SAVE_DIR}/bert_model")

print("Models saved successfully in Google Drive")

In [ ]:
def predict_emotion(text):
    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(bert_model.device) for key, value in inputs.items()}

    bert_model.eval()

    with torch.no_grad():
        outputs = bert_model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()

    emotion = emotion_labels[prediction]
    return emotion

In [ ]:
def therapist_response(emotion):
    responses = {
        "sadness": "I'm sorry you're feeling this way. Would you like to tell me what has been making you feel sad?",
        "fear": "That sounds scary. Take a slow breath. What is making you feel afraid right now?",
        "nervousness": "It makes sense to feel nervous. What part of the situation worries you the most?",
        "anger": "I can hear that you're upset. Would you like to talk about what triggered this feeling?",
        "joy": "I'm glad you're feeling good. What made you happy today?",
        "love": "That sounds meaningful. Tell me more about this feeling.",
        "confusion": "It sounds like things feel unclear. Let's take it one step at a time.",
        "disappointment": "That sounds disappointing. It can hurt when things do not go as expected.",
        "remorse": "It sounds like you're carrying regret. Would it help to talk through what happened?",
        "neutral": "I'm here with you. Tell me more about what is on your mind."
    }

    return responses.get(
        emotion,
        "Thank you for sharing that with me. Would you like to tell me more?"
    )

In [ ]:
user_text = "I feel very stressed because of exams."

predicted_emotion = predict_emotion(user_text)
bot_reply = therapist_response(predicted_emotion)

print("User:", user_text)
print("Predicted Emotion:", predicted_emotion)
print("Bot:", bot_reply)

In [ ]:
conversation_history = []

while True:
    user_text = input("You: ")

    if user_text.lower() in ["exit", "quit", "stop"]:
        print("Bot: Take care. I'm glad you talked with me.")
        break

    predicted_emotion = predict_emotion(user_text)
    bot_reply = therapist_response(predicted_emotion)

    conversation_history.append({
        "user": user_text,
        "emotion": predicted_emotion,
        "bot": bot_reply
    })

    print("Emotion:", predicted_emotion)
    print("Bot:", bot_reply)

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    bert_labels,
    bert_preds,
    target_names=emotion_labels,
    zero_division=0
)

print(report)


In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/AI_Therapist_Results"
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(f"{RESULTS_DIR}/classification_report.txt", "w") as file:
    file.write(report)

results.to_csv(f"{RESULTS_DIR}/model_comparison.csv", index=False)

print("Results saved successfully")

In [ ]:
!pip install gradio -q

In [ ]:
import torch
import pickle
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR = "/content/drive/MyDrive/AI_Therapist_Models"
BERT_MODEL_PATH = f"{SAVE_DIR}/bert_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_PATH)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_PATH
)

bert_model.to(device)
bert_model.eval()

with open(f"{SAVE_DIR}/emotion_labels.pkl", "rb") as file:
    emotion_labels = pickle.load(file)

print("BERT model and labels loaded successfully")
print("Number of labels:", len(emotion_labels))

In [ ]:
def predict_emotion(text):
    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()

    emotion = emotion_labels[prediction]
    return emotion

In [ ]:
def therapist_response(emotion, user_text=""):
    text = user_text.lower()

    if any(word in text for word in ["stress", "stressed", "exam", "exams", "test", "tests", "pressure"]):
        return "It makes sense to feel stressed, especially when exams are involved. What part feels most overwhelming right now?"

    responses = {
        "sadness": "I'm sorry you're feeling this way. Would you like to tell me what has been making you feel sad?",
        "fear": "That sounds scary. Take a slow breath. What is making you feel afraid right now?",
        "nervousness": "It makes sense to feel nervous. What part of the situation worries you the most?",
        "anger": "I can hear that you're upset. Would you like to talk about what triggered this feeling?",
        "joy": "I'm glad you're feeling good. What made you happy today?",
        "love": "That sounds meaningful. Tell me more about this feeling.",
        "confusion": "It sounds like things feel unclear. Let's take it one step at a time.",
        "disappointment": "That sounds disappointing. It can hurt when things do not go as expected.",
        "remorse": "It sounds like you're carrying regret. Would it help to talk through what happened?",
        "neutral": "I'm here with you. Tell me more about what is on your mind."
    }

    return responses.get(
        emotion,
        "Thank you for sharing that with me. Would you like to tell me more?"
    )

In [ ]:
user_text = "I feel very stressed because of exams."

predicted_emotion = predict_emotion(user_text)
bot_reply = therapist_response(predicted_emotion, user_text)

print("User:", user_text)
print("Predicted Emotion:", predicted_emotion)
print("Bot:", bot_reply)

In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr

conversation_memory = []

def chat_with_saved_model(message, history):
    if message is None or message.strip() == "":
        return history, ""

    predicted_emotion = predict_emotion(message)
    bot_reply = therapist_response(predicted_emotion, message)

    conversation_memory.append({
        "user": message,
        "emotion": predicted_emotion,
        "bot": bot_reply
    })

    final_reply = f"Predicted Emotion: {predicted_emotion}\n\n{bot_reply}"

    history.append((message, final_reply))

    return history, ""

def clear_chat():
    conversation_memory.clear()
    return [], ""

with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# AI Therapist Chatbot with Emotion Detection")
    gr.Markdown("This chatbot uses your saved BERT model weights from Google Drive.")

    chatbot = gr.Chatbot(
        label="Chatbot",
        height=450
    )

    message = gr.Textbox(
        label="Your Message",
        placeholder="Type how you are feeling...",
        lines=2
    )

    with gr.Row():
        send_btn = gr.Button("Send")
        clear_btn = gr.Button("Clear Chat")

    send_btn.click(
        fn=chat_with_saved_model,
        inputs=[message, chatbot],
        outputs=[chatbot, message]
    )

    message.submit(
        fn=chat_with_saved_model,
        inputs=[message, chatbot],
        outputs=[chatbot, message]
    )

    clear_btn.click(
        fn=clear_chat,
        inputs=[],
        outputs=[chatbot, message]
    )

app.launch(share=True, debug=True)